## Convolutional Neural Network using physicochemical descriptors - testing performance at different protein similarity thresholds

### Load the data:

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import json
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, ParameterGrid
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score

Xtrain = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/Xtrainfeatnew.csv', index_col=0)
Xtest = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/Xtestfeatnew.csv', index_col=0)
X = pd.concat([Xtrain, Xtest])
X = X.sort_values(by='Index_original')
X_csv = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/refined-set-csv.csv')

X_scaled = X.values
scaler = StandardScaler()
scaler.fit(X_scaled)
X_scaled = scaler.transform(X_scaled)
X_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

data = X_scaled.join(X_csv['PDB_Code'])
data = data.join(X_csv['Log_binding'])
display(data)

,MolecularWeight,Aromaticity,InstabilityIndex,Gravy,MoreauBrotoAuto_Hydrophobicity1,MoreauBrotoAuto_Hydrophobicity2,MoreauBrotoAuto_Hydrophobicity3,MoreauBrotoAuto_Hydrophobicity4,MoreauBrotoAuto_Hydrophobicity5,MoreauBrotoAuto_Hydrophobicity6,...,MolLogP,NumHAcceptors,NumHDonors,NumRotatableBonds,RingCount,LabuteASA,Kappa1,BalabanJ,PDB_Code,Log_binding
Index_original,,,,,,,,,,,,,,,,,,,,,
0,-0.403247,0.443234,-1.439150,-1.227274,-0.524876,-0.498692,-0.443167,-0.462878,-0.479052,-0.436454,...,0.285294,-0.797962,-0.409886,-1.206585,-0.403346,-0.980527,-1.061284,0.692051,6ugp,6.16
1,-0.652511,0.788103,-0.031606,-1.527422,1.540581,1.595904,1.701885,1.710882,1.722799,1.478996,...,-0.508946,0.664238,0.417243,-0.623668,0.208139,-0.577759,-0.654069,-0.216307,4rdn,5.92
2,0.577037,-1.308488,0.837897,1.086976,0.227358,0.220924,0.215855,0.213982,0.194455,0.188999,...,-0.735889,1.882737,0.003679,-0.040751,0.208139,-0.039545,0.086547,-0.570254,4mo4,4.22
3,-0.734201,-1.962922,0.236164,-0.489112,-0.091385,-0.241686,-0.094273,0.031750,0.155599,0.071727,...,1.297098,-1.285362,-0.823451,-0.817974,0.208139,-0.758488,-0.996441,-0.176129,3s0b,6.49
4,-0.770521,1.834490,0.215461,0.591285,0.418604,0.400828,0.396763,0.357163,0.194455,0.175969,...,0.037965,-0.554262,-0.409886,-0.623668,-0.403346,-0.721590,-0.767975,-0.516917,6r1d,5.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5244,1.770321,-0.675112,0.497581,0.691613,0.189109,0.195223,0.215855,0.213982,0.207407,0.188999,...,0.590321,-0.554262,-0.409886,-0.623668,0.819624,-0.007462,-0.250521,-0.493753,6p3t,7.32
5245,-0.523628,-0.656959,-1.232828,0.792916,0.954093,0.901988,0.900721,0.916874,0.997483,0.931725,...,-0.637656,-1.041662,-0.409886,-0.817974,-1.014830,-1.105159,-0.990943,1.099011,1tx7,4.60
5246,-0.252303,-2.286593,-1.252650,0.777004,0.597100,0.593582,0.590593,0.591461,0.583017,0.579908,...,-0.820009,2.126437,0.003679,-0.040751,0.208139,-0.070280,0.052949,-0.558932,3ta1,3.25


In [3]:
import random
import numpy as np
def NRKFold(E,pc,K = 5, shuffle=True):
    """
    Generate non-redundant K-folds for a dataset where each example involves an object 
    that belongs to a certain cluster. This function ensures that no two folds contain 
    objects from the same cluster and aims to distribute the number of examples 
    approximately equally across all folds.

    This is particularly useful in scenarios where data points can naturally group into 
    clusters (e.g., proteins in bioinformatics), and it is important to avoid having 
    similar examples in both the training and validation sets of a particular fold.

    Parameters
    ----------
    E : List
        A list containing identifiers of objects involved in each example.
    pc : Dictionary
        A dictionary mapping each object to its cluster assignment.
    K : Integer, optional
        The number of folds to create. Default is 5.
    shuffle : Boolean, optional
        Determines whether to shuffle the cluster to fold assignments in different runs.
        Default is True.

    Returns
    -------
    List of lists
        A list where each sublist contains the indices of examples in `objects` that belong to a particular fold.

    Example
    -------
    >>> objects = ['obj1', 'obj2', 'obj3', 'obj4', 'obj5', 'obj6', 'obj1']
    >>> clusters = {'obj1': 1, 'obj2': 2, 'obj3': 1, 'obj4': 2, 'obj5': 3, 'obj6': 3}
    >>> folds = NRKFold(objects, clusters, K=2, shuffle=False)
    >>> print(folds)
    Output might be: [[0, 2, 6], [1, 3, 4, 5]]
    Here, objects 'obj1', 'obj3', and 'obj1' (indices 0, 2, 6) are in one fold, 
    and the rest are in another fold, ensuring no fold has objects from the same cluster.
    """
    e = [pc[str(x)] for x in E] #cluster indices of all proteins in the examples
    c2idx={} #indices of examples of each cluster in e
    for i,x in enumerate(e):
        try: 
            c2idx[x].append(i)
        except:
            c2idx[x]=[i]    
    ce = dict([(c,len(c2idx[c])) for c in c2idx]) #counts of examples of different clusters    
    cF = [0]*K; #counts of examples in each fold
    CF = [[] for _ in range(K)]; #clusters in each fold
    F = [[] for _ in range(K)];#indices of examples in each fold
    keys = list(ce.keys())
    if shuffle:
        random.shuffle(keys)
    for k in keys:
        v = ce[k]
        idx = np.argmin(cF)
        cF[idx]+=v
        CF[idx].append(k) #add cluster to fold
        F[idx].extend(c2idx[k])
    return F

In [48]:
with open("/dcs/22/u2243582/cs310/nrkf_physicochem_feat/clustered_proteins.json", "r") as json_file:
    cluster_dict = json.load(json_file)

print(len(cluster_dict)) # length of dict should match the output of num of clusters from clustering program

cluster_assignment_dict = {}
for cluster_ind in cluster_dict:
    for pdb in cluster_dict[cluster_ind]:
        cluster_assignment_dict[pdb] = cluster_ind
        
identifiers = list(cluster_assignment_dict.keys())

1327


In [49]:
folds = NRKFold(identifiers, cluster_assignment_dict)

In [50]:
final_folds = []
for fold in folds:
    temp_fold = []
    for obj in fold:
        pdb = identifiers[obj]
        data_ind = data.index[data['PDB_Code'] == pdb].item()
        temp_fold.append(data_ind)
    final_folds.append(temp_fold)

In [51]:
# The final NRKfolds:
splits = [
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[4]
    },
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[4],
        "test_ix": final_folds[3]
    },
    {
       "train_ix": final_folds[0] + final_folds[1] + final_folds[4] + final_folds[3],
        "test_ix": final_folds[2]
    },
    {
        "train_ix": final_folds[0] + final_folds[4] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[1]
    },
    {
        "train_ix": final_folds[4] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[0]
    }
]

### CNN:

In [7]:
# Detect GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
class CNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_filters, kernel_size, dropout_rate):
        super(CNNModel, self).__init__()

        self.hidden_dim = hidden_dim  

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(num_filters)

        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters * 2, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(num_filters * 2)

        self.conv3 = nn.Conv1d(in_channels=num_filters * 2, out_channels=num_filters * 4, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn3 = nn.BatchNorm1d(num_filters * 4)

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)  # Dropout layer

        self.flatten_dim = self._get_flatten_dim(input_dim, num_filters, kernel_size)
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.flatten_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)  # New fully connected layer
        self.fc3 = nn.Linear(hidden_dim // 2, 1)

    def _get_flatten_dim(self, input_dim, num_filters, kernel_size):
        # Helper function to compute the output size dynamically
        
        sample_input = torch.randn(1, 1, input_dim)  
        
        x = self.pool(self.relu(self.bn1(self.conv1(sample_input))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        return x.view(1, -1).size(1)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        x = x.view(x.size(0), -1)  # Flatten tensor to [batch_size, flattened_dim]

        x = self.relu(self.fc1(x))
        x = self.dropout(x) 

        x = self.relu(self.fc2(x))
        # x = self.dropout(x) 

        x = self.fc3(x)

        return x.squeeze()

In [9]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs=50, patience=10):
    
    model.to(device)
    best_loss = float("inf")
    patience_counter = 0
    epoch_losses = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        epoch_losses.append(avg_train_loss)

        # ---- Validation Step ----
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                val_loss += criterion(y_pred, y_batch).item()

        avg_val_loss = val_loss / len(val_loader)
        
        # ---- Early Stopping Check ----
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping after {epoch+1} epochs")
                break  

    return epoch_losses

In [15]:
def cross_validate(data, params, n_splits=5, batch_size=32, epochs=50):
    
    all_loss_curves = []
    count = 1

    results = []  # Store results

    for fold in splits:
        print(f"Fold {count}")
        
        train_ix = np.array(fold["train_ix"])
        test_ix = np.array(fold["test_ix"])
        
        X_train = data.loc[train_ix, 'MolecularWeight':'BalabanJ'].values
        X_test = data.loc[test_ix, 'MolecularWeight':'BalabanJ'].values
        y_train = data.loc[train_ix, 'Log_binding'].values
        y_test = data.loc[test_ix, 'Log_binding'].values #.reshape(-1,1)
        
        X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
        y_train = torch.tensor(y_train, dtype=torch.float32)
        y_test = torch.tensor(y_test, dtype=torch.float32)

        # Train/Validation Split 
        train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

        # Initialize model
        model = CNNModel(input_dim=285, 
                hidden_dim=params["hidden_dim"],
                num_filters=params["num_filters"],
                kernel_size=params["kernel_size"],
                dropout_rate=params["dropout_rate"]).to(device)
        criterion = nn.L1Loss()
        optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=0)

        # Train model
        epoch_losses = train(model, train_loader, test_loader, criterion, optimizer, epochs=epochs)
        all_loss_curves.append(epoch_losses)

        # ---- Evaluate Model ----
        model.eval()
        y_pred_list, y_true_list = [], []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_pred = model(X_batch).squeeze().cpu().numpy()
                y_true = y_batch.squeeze().cpu().numpy()

                y_pred_list.extend(y_pred)
                y_true_list.extend(y_true)

        pearson_corr, pearson_p = pearsonr(y_true_list, y_pred_list)
        spearman_corr, spearman_p = spearmanr(y_true_list, y_pred_list)
        mae = mean_absolute_error(y_true_list, y_pred_list)
        variance = np.var(np.array(y_true_list) - np.array(y_pred_list))
        r2 = r2_score(y_true_list, y_pred_list)

        results.append({"Fold": count, "Pearson": pearson_corr, "Pearson p": pearson_p, "Spearman": spearman_corr, "Spearman p": spearman_p, "MAE": mae, "Variance": variance, "R2": r2})
        count +=1
        
    return results

In [52]:
# Use modal hyperparameters from baseline nested cv:
params = {'hidden_dim': 256, 'dropout_rate': 0.1, 'learning_rate': 0.01, 'num_filters': 64, 'kernel_size': 7}

results = cross_validate(data, params, n_splits=5, batch_size=64, epochs=50)

Fold 1
Early stopping after 24 epochs
Fold 2
Early stopping after 22 epochs
Fold 3
Early stopping after 20 epochs
Fold 4
Early stopping after 15 epochs
Fold 5
Early stopping after 19 epochs


In [53]:
# Nested CV Extract metrics
pearson_scores = [result['Pearson'] for result in results]
spearman_scores = [result['Spearman'] for result in results]
mae_scores = [result['MAE'] for result in results]
var_scores = [result['Variance'] for result in results]
r2_scores = [result['R2'] for result in results]
pearson_p_values = [result['Pearson p'] for result in results]
spearman_p_values = [result['Spearman p'] for result in results]

# Calculate means and standard deviations
pearson_mean = np.mean(pearson_scores)
pearson_std = np.std(pearson_scores)

spearman_mean = np.mean(spearman_scores)
spearman_std = np.std(spearman_scores)

mae_mean = np.mean(mae_scores)
mae_std = np.std(mae_scores)

var_mean = np.mean(var_scores)
var_std = np.std(var_scores)

r2_mean = np.mean(r2_scores)
r2_std = np.std(r2_scores)

pearson_p_mean = np.mean(pearson_p_values)
pearson_p_std = np.std(pearson_p_values)

spearman_p_mean = np.mean(spearman_p_values)
spearman_p_std = np.std(spearman_p_values)

# Print the results
print(f"Pearson Correlation: Mean = {pearson_mean}, Std = {pearson_std}, p-value Mean = {pearson_p_mean}, p-value Std = {pearson_p_std}")
print(f"Spearman Correlation: Mean = {spearman_mean}, Std = {spearman_std}, p-value Mean = {spearman_p_mean}, p-value Std = {spearman_p_std}")
print(f"MAE: Mean = {mae_mean}, Std = {mae_std}")
print(f"Variance: Mean = {var_mean}, Std = {var_std}")
print(f"R2: Mean = {r2_mean}, Std = {r2_std}")

Pearson Correlation: Mean = 0.5188061784561484, Std = 0.06368079829586487, p-value Mean = 1.9984978175112347e-50, p-value Std = 3.9889524303420076e-50
Spearman Correlation: Mean = 0.5281363438512242, Std = 0.07619607373195852, p-value Mean = 3.6922078971294695e-49, p-value Std = 7.064791653224904e-49
MAE: Mean = 1.4993102550506592, Std = 0.10718662291765213
Variance: Mean = 2.7459075450897217, Std = 0.3364814221858978
R2: Mean = 0.052748701623323214, Std = 0.14283303465932037
